# Jensen Alpha Momentum — 12M — Winner Drift

One signal, one formation horizon and one maintenance method. The experiment contains nine cells: rebalance every 1, 3 or 6 months × target N=12, 24 or 50.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))
from momentum_india.notebook_views import ResearchNotebook

research = ResearchNotebook('jensen_alpha', '12M', 'winner_drift')

## 1. Signal and portfolio rule

Daily stock excess return = alpha + beta × daily Nifty 50 excess return + residual. Both excess returns subtract the continuous liquid-fund daily return. OLS alpha is the ranking score over the fixed formation window; this is a trailing signal, not the strategy's subsequently realized alpha.

Continuing holdings retain their naturally drifted weights, capped at 2/N at scheduled rebalances. Exits and entrants are paired in stable symbol order; an entrant receives min(exit weight, 1/N). Excess exit weight and cap trims are spread equally among continuing holdings with room below the cap. Unmatched entrants share existing cash up to 1/N each. Residual cash is retained. The cap is a scheduled target constraint, not a daily trim rule.

## 2. Universe → ranking → actual portfolio

The example uses the latest monthly N=24 signal and exposes the ranking inputs and actual target weights.

In [2]:
research.snapshot()

Symbol,Daily alpha,MDTV (INR)
NSE:CUPID-EQ,0.0090,"2,261,224,535.95"
NSE:ATHERENERG-EQ,0.0057,"2,745,338,022.88"
NSE:KMEW-EQ,0.0054,"175,654,741.20"
NSE:THANGAMAYL-EQ,0.0048,"572,585,716.40"
NSE:RPTECH-EQ,0.0047,"78,147,051.55"
NSE:HFCL-EQ,0.0045,"3,688,024,397.99"
NSE:NGLFINE-EQ,0.0044,"21,228,409.00"
NSE:AEROFLEX-EQ,0.0043,"644,982,030.67"
NSE:SILVERTUC-EQ,0.0043,"65,229,764.81"
NSE:APOLLO-EQ,0.0042,"2,231,391,532.53"


Symbol,Leg,Actual weight,Current selection,Execution status,Daily alpha,MDTV (INR)
NSE:CUPID-EQ,long,2.8%,True,selected,0.0090,"2,261,224,535.95"
NSE:ATHERENERG-EQ,long,0.3%,True,selected,0.0057,"2,745,338,022.88"
NSE:KMEW-EQ,long,0.7%,True,selected,0.0054,"175,654,741.20"
NSE:THANGAMAYL-EQ,long,1.0%,True,selected,0.0048,"572,585,716.40"
NSE:RPTECH-EQ,long,0.6%,True,selected,0.0047,"78,147,051.55"
NSE:HFCL-EQ,long,0.5%,True,selected,0.0045,"3,688,024,397.99"
NSE:NGLFINE-EQ,long,0.2%,True,selected,0.0044,"21,228,409.00"
NSE:AEROFLEX-EQ,long,0.6%,True,selected,0.0043,"644,982,030.67"
NSE:SILVERTUC-EQ,long,0.2%,True,selected,0.0043,"65,229,764.81"
NSE:APOLLO-EQ,long,0.7%,True,selected,0.0042,"2,231,391,532.53"


Symbol,Formation start,Start adjusted close,Signal close date,End adjusted close,Formation price return
NSE:CUPID-EQ,2025-07-31,30.18,2026-07-31,230.62,664.2%


## 3. Return layers across all nine cells

Raw is before trading charges. After-cost gross/pre-tax deducts modeled trading charges. The long-only post-tax overlay additionally applies the annual equity-gains ledger. The academic reference instead compares raw and borrowing-adjusted layers.

In [3]:
research.layers_bridge()

Rebalance,First date,Last date,Sessions
1M,2007-05-03,2026-08-28,4771
3M,2007-07-02,2026-08-28,4729
6M,2007-07-02,2026-08-28,4729


Rebalance,N,Raw,After costs,Post-tax overlay
1M,12,9.8%,9.4%,8.7%
1M,24,12.3%,11.9%,11.2%
1M,50,13.9%,13.4%,12.5%
3M,12,13.0%,12.8%,11.5%
3M,24,12.0%,11.8%,11.0%
3M,50,12.6%,12.3%,11.5%
6M,12,12.4%,12.2%,11.0%
6M,24,12.4%,12.1%,10.8%
6M,50,10.8%,10.6%,9.9%


## 4. Risk-adjusted results

Sharpe uses daily excess returns relative to the liquid fund. VaR and expected shortfall are historical monthly 95% loss measures. Partial first/last months are included. Time below prior peak counts days awaiting a new all-time high—not losing days. The initial invested capital is included as the first peak.

In [4]:
research.risk_grid()

Rebalance,N,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,12,8.7%,0.173,-80.7%,9.3%
1M,24,11.2%,0.310,-76.7%,7.6%
1M,50,12.5%,0.374,-75.2%,8.6%
3M,12,11.5%,0.302,-75.7%,10.1%
3M,24,11.0%,0.289,-74.7%,8.1%
3M,50,11.5%,0.318,-75.8%,9.5%
6M,12,11.0%,0.280,-71.3%,9.3%
6M,24,10.8%,0.276,-70.8%,9.9%
6M,50,9.9%,0.235,-75.7%,8.2%


Rebalance,N,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,12,16.7%,93.5%,1944
1M,24,15.0%,91.7%,1643
1M,50,15.6%,91.8%,1649
3M,12,16.7%,93.7%,1579
3M,24,15.3%,92.3%,1601
3M,50,15.9%,93.2%,1637
6M,12,16.4%,94.7%,1601
6M,24,15.8%,94.2%,1551
6M,50,16.1%,93.5%,1598


### CAGR

In [5]:
research.heatmap('cagr')

### Sharpe ratio

In [6]:
research.heatmap('sharpe')

### Maximum drawdown

In [7]:
research.heatmap('maximum_drawdown')

### Monthly 95% VaR

In [8]:
research.heatmap('monthly_var_95')

## 5. Equity paths and matched risks

Each chart fixes breadth and compares rebalance frequencies. Final-layer curves and the price benchmark start visible; other return layers remain in the selectable legend. Logarithmic axes make early and late periods comparable; the bottom range slider preserves the full history. Curves display weekly observations for readability, while every statistic uses the complete daily series.

### N=12

In [9]:
research.equity(12)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,9.8%,0.224,-79.7%,9.1%
1M,After costs,9.4%,0.204,-80.0%,9.3%
1M,Post-tax overlay,8.7%,0.173,-80.7%,9.3%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,13.0%,0.370,-74.4%,9.9%
3M,After costs,12.8%,0.358,-74.6%,10.1%
3M,Post-tax overlay,11.5%,0.302,-75.7%,10.1%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,12.4%,0.347,-69.5%,9.2%
6M,After costs,12.2%,0.335,-69.6%,9.3%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,16.3%,92.6%,1781
1M,After costs,16.4%,93.0%,1785
3M,Raw,16.3%,92.9%,1568
3M,After costs,16.4%,93.0%,1570
6M,Raw,15.9%,94.0%,1581
6M,After costs,16.1%,94.1%,1582
1M,Post-tax overlay,16.7%,93.5%,1944
3M,Post-tax overlay,16.7%,93.7%,1579
6M,Post-tax overlay,16.4%,94.7%,1601
1M,Nifty 50 price,13.1%,92.4%,1520


### N=24

In [10]:
research.equity(24)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,12.3%,0.373,-76.3%,7.4%
1M,After costs,11.9%,0.348,-76.6%,7.6%
1M,Post-tax overlay,11.2%,0.310,-76.7%,7.6%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,12.0%,0.344,-74.1%,7.9%
3M,After costs,11.8%,0.330,-74.3%,8.1%
3M,Post-tax overlay,11.0%,0.289,-74.7%,8.1%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,12.4%,0.351,-69.9%,9.9%
6M,After costs,12.1%,0.335,-70.1%,9.9%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,14.9%,90.6%,1600
1M,After costs,15.0%,91.1%,1642
3M,Raw,15.1%,91.1%,1598
3M,After costs,15.2%,91.3%,1600
6M,Raw,15.6%,93.5%,1530
6M,After costs,15.6%,93.8%,1541
1M,Post-tax overlay,15.0%,91.7%,1643
3M,Post-tax overlay,15.3%,92.3%,1601
6M,Post-tax overlay,15.8%,94.2%,1551
1M,Nifty 50 price,13.1%,92.4%,1520


### N=50

In [11]:
research.equity(50)

Rebalance,Return layer,CAGR,Sharpe,Max drawdown,Monthly VaR 95%
1M,Raw,13.9%,0.446,-74.6%,8.6%
1M,After costs,13.4%,0.421,-74.8%,8.6%
1M,Post-tax overlay,12.5%,0.374,-75.2%,8.6%
1M,Nifty 50 price,9.7%,0.219,-59.9%,6.9%
3M,Raw,12.6%,0.375,-75.0%,9.5%
3M,After costs,12.3%,0.359,-75.2%,9.5%
3M,Post-tax overlay,11.5%,0.318,-75.8%,9.5%
3M,Nifty 50 price,9.5%,0.209,-59.9%,7.0%
6M,Raw,10.8%,0.281,-74.9%,8.2%
6M,After costs,10.6%,0.269,-75.0%,8.2%


Rebalance,Return layer,Monthly ES 95%,Time below prior peak,Longest underwater (sessions)
1M,Raw,15.3%,90.7%,1639
1M,After costs,15.5%,91.2%,1646
3M,Raw,15.6%,92.2%,1597
3M,After costs,15.7%,92.3%,1601
6M,Raw,15.9%,92.8%,1591
6M,After costs,15.9%,93.0%,1594
1M,Post-tax overlay,15.6%,91.8%,1649
3M,Post-tax overlay,15.9%,93.2%,1637
6M,Post-tax overlay,16.1%,93.5%,1598
1M,Nifty 50 price,13.1%,92.4%,1520


## 6. Recovery burden

The longest underwater episode is shown by its peak, trough and recovery dates. Unrecovered episodes remain explicitly open.

In [12]:
research.recovery()

Series,Peak,Trough,Recovery,Sessions,Calendar days,Episode loss
1M,2008-01-07,2009-03-09,2014-09-05,1643,2432,-76.7%
3M,2008-01-07,2009-03-09,2014-07-04,1601,2369,-74.7%
6M,2008-01-07,2009-03-09,2014-04-23,1551,2297,-70.8%
Nifty · 1M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 3M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%
Nifty · 6M,2008-01-08,2008-10-27,2014-03-06,1520,2248,-59.9%


## 7. Portfolio behavior

Scheduled turnover is (buy value + sell value)/(2 × pre-trade equity). Retention compares successive scheduled target name sets. Cash and total charges include the intervening daily path.

In [13]:
research.behavior()

Rebalance,N,Mean names at rebalance,Mean cash weight,Scheduled name retention
1M,12,17.09,36.6%,73.1%
1M,24,35.37,35.2%,76.5%
1M,50,72.17,14.5%,78.5%
3M,12,17.23,20.7%,54.2%
3M,24,33.88,29.1%,57.5%
3M,50,69.75,17.7%,61.6%
6M,12,16.00,21.0%,36.5%
6M,24,33.31,8.9%,42.6%
6M,50,68.51,18.5%,46.2%


Rebalance,N,Mean scheduled turnover,All trading charges (INR),Days with stop sales
1M,12,15.1%,"1,424,635.98",0
1M,24,13.6%,"1,953,896.13",0
1M,50,15.8%,"3,071,907.37",0
3M,12,33.8%,"1,834,047.30",0
3M,24,27.1%,"1,239,452.93",0
3M,50,29.9%,"1,668,614.96",0
6M,12,47.5%,"1,432,122.54",0
6M,24,50.9%,"1,329,554.15",0
6M,50,43.1%,"1,011,945.47",0


## 8. Market-state attribution

This is an observation, not an extra strategy filter. Prior-close Nifty 50 versus SMA(200) defines up/down; 63-session volatility versus its expanding historical median defines high/low volatility. The representative monthly N=24 path is shown with shaded states.

In [14]:
research.regimes()

Market state,Sessions,Mean daily return,Daily volatility,Positive days
Down / High volatility,748,-0.07%,1.46%,54.28%
Down / Low volatility,609,0.02%,0.93%,53.20%
Up / High volatility,720,0.17%,1.40%,60.83%
Up / Low volatility,2694,0.05%,0.79%,56.64%


## 9. Complete portfolio and trade evidence

Separate CSV files retain all scheduled portfolios, actual trades and risk layers for this exact signal/lookback/maintenance combination.

In [15]:
research.portfolio_exports()

Rebalance,N,First rebalance,Last rebalance,Rebalance dates,Holding rows
1M,12,2007-05-03,2026-08-03,232,3964
1M,24,2007-05-03,2026-08-03,232,8206
1M,50,2007-05-03,2026-08-03,232,16743
3M,12,2007-07-02,2026-07-01,77,1327
3M,24,2007-07-02,2026-07-01,77,2609
3M,50,2007-07-02,2026-07-01,77,5371
6M,12,2007-07-02,2026-07-01,39,624
6M,24,2007-07-02,2026-07-01,39,1299
6M,50,2007-07-02,2026-07-01,39,2672


## Findings

In [16]:
research.conclusion()

Matched reference: [Raw Momentum — 12M — Winner Drift](05_raw_momentum_12m_winner_drift.ipynb). The comparison holds formation, maintenance, frequency and N fixed.